In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler

In [ ]:
#Importamos archivo con librerias
df_puerto = pd.read_csv('datosPuerto.csv')
df_puerto

In [ ]:
#Un describe para ver que datos tenemos
df_puerto.describe()

In [ ]:
#Hacemos un info para ver los tipos de datos y los NaN
df_puerto.info()

In [ ]:
#Pasamos fecha a datetime
df_puerto["FECHA"] = pd.to_datetime(df_puerto["FECHA"])
df_puerto['FECHA'] = df_puerto['FECHA'].dt.year

In [ ]:
df_puerto_paisesbajos = df_puerto[
    (df_puerto['PAIS_ORIGEN'].isin(['NL', 'ES'])) &
    (df_puerto['PAIS_FINAL'].isin(['NL', 'ES'])) &
    (df_puerto['PAIS_ORIGEN'] != df_puerto['PAIS_FINAL'])
].copy()


In [ ]:
#creamos columna actividad total para ver la actividad del puerto
puerto_agrupado = df_puerto_paisesbajos.groupby('FECHA').agg(Peso_Total=('PESO_KILOS', 'sum'),Actividad_Total=('PESO_KILOS', 'count') ).reset_index()
puerto_agrupado

In [ ]:
#CARGAMOS MACRO ECONOMICAS
df_variables = pd.read_csv("world_bank_country_data.csv")

In [ ]:
#Filtramos
df_variables_paisesbajos = df_variables[df_variables['country'] == 'Netherlands'].copy()

In [ ]:
#Hacemos pivot table
df_pivot_table = df_variables_paisesbajos.pivot_table(index='date', columns='indicator', values='value').sort_index()
df_pivot_table

In [ ]:
df_pivot_table.isna().mean()*100

In [ ]:
#Y hacemos merge
df_juntado = pd.merge(puerto_agrupado, df_pivot_table, left_on='FECHA', right_index=True, how='inner')
df_juntado

In [ ]:
#Creamos la columna actividad año siguiente y miramos si aumenta o disminuye con (1,0)
df_juntado['Actividad_Año_siguiente'] = df_juntado['Actividad_Total']
df_juntado['Aumenta'] = (df_juntado['Actividad_Año_siguiente'] > df_juntado['Actividad_Total']).astype(int)
df_juntado

In [ ]:
#Rellenamos los NaN con la mediana
df_clase = df_clase.fillna(df_clase.median())
df_clase

In [ ]:
#Importamos librerias y usamos
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer


cols_usar = [
    'Peso_Total','Actividad_Total',
    'Agricultural raw materials exports (% of merchandise exports)',
    'Commercial service exports (current US$)',
    'Export value index (2015 = 100)',
    'Exports of goods and services (% of GDP)',
    'External balance on goods and services (current US$)',
    'Food imports (% of merchandise imports)',
    'Fuel exports (% of merchandise exports)',
    'Merchandise imports (current US$)',
    'Merchandise trade (% of GDP)',
    'Net food imports by crop - Barley',
    'Nominal Effecive Exchange Rate',
    'Population growth (annual %)',
    'Total electricity output (GWh)',
    'Trade (% of GDP)']


In [ ]:
# Variables
X = df_clase[cols_usar]
y = df_clase['Aumenta']

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# Escalado robusto para el modelo de clasificación (SVC)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Hacemos el grid search con los datos escalados
param_grid = {"C": np.arange(0.0001, 100)}
svm = SVC()
svm_cv = GridSearchCV(svm, param_grid, cv=5, scoring="recall")
svm_cv.fit(X_train_scaled, y_train) 

In [ ]:
#Printeamos mejor parametro y score
print("Mejor parametro:", svm_cv.best_params_)
print("Mejor score:", svm_cv.best_score_)

In [ ]:
# Miramos y entrenamos el mejor estimador
best_svm = svm_cv.best_estimator_
best_svm.fit(X_train_scaled, y_train) 

In [ ]:
# Predecimos, y nos da que si que aumenta
pred_2024 = best_svm.predict(X_test_scaled) 
pred_2024